# Detect Columns by using ML

---

In [ ]:
import numpy as np
import pandas as pd
import cv2
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
from functions import *
import scipy
import signal
import io
import cv2
import os
import glob
from scipy.signal import find_peaks
from joblib import Parallel, delayed
from tqdm.notebook import tqdm

**Machine learning (ML)** is a branch of artificial intelligence. It allows a computer to learn from data and to improve decision making with experience.

---

## Using Random Forest

**L'Arbre de Décision (Decision Tree) :**
Imagine un jeu de "Qui est-ce ?". L'algorithme pose une série de questions par oui/non sur les caractéristiques (features) de tes données pour arriver à une conclusion. Par exemple : "La variance de cette colonne est-elle supérieure à 450 ?" -> Si oui, on va à droite ; si non, on va à gauche.

Le Random Forest repose sur l'apprentissage d'ensemble (Ensemble Learning), et plus précisément sur une technique appelée Bagging (Bootstrap Aggregating). L'algorithme va créer une "forêt" composée de dizaines, voire de centaines d'arbres de décision.

Pour classer une nouvelle colonne (Normale vs Défectueuse), la forêt fait passer les données de la colonne dans tous ses arbres. Chaque arbre vote. La classe qui obtient la majorité des votes l'emporte.

In [ ]:
def load_images(folder='train',type='VGA',sequence='sequence_1', dyn='low dyn with columns 1', force_gray=False):
    """
    function that load a folder of images

    args : 
    folder : folder type
    type : type of the frame (HD, VGA, SXGA)
    sequence : sequence_1, sequence_2, sequence_3
    dyn : low dyn with columns 1, low dyn with columns 2, low dyn with columns 3
    force_gray : bool to load in gray (one array)

    return :
    list of the images of the folder
    """
    images = []

    chemin_recherche = os.path.join(folder, type, sequence, dyn, '*.png')

    fichiers_trouves = sorted(glob.glob(chemin_recherche))

    if force_gray == True:
        for image_path in fichiers_trouves:
            img = cv2.imread(image_path, as_gray=True)
            images.append(img)
    else:
        for image_path in fichiers_trouves:
            img = cv2.imread(image_path)
            images.append(img)
            
    return images

### Features that we use : 

- Moyenne `mean` : Valeur moyenne des intensités de la colonne. Luminosité globale de la colonne. Si trop ou trop basse peut indiquer anomalie.
- Ecart-type `std` : Dispersion autour de la moyenne. Colonne peut-être *noisy* si l'écart-type est très élevé.
- Min et Max : Si la colonne à des valeurs super hautes ou super basses c'est que l'amplificateur est défectueux (dixit Phlypo)
- Range `range` : Différence entre max-min
- Médian `median` : Valeur centrale. Comparaison avec la moyenne intéressant.
- Quantile `q25` `q75` : Quartile.
- Skewness `skewness` : Asymétrie 
- Kurtosis `kurtosis`: Applatissement 
- Energie `energy` : Energie d'un signal (dixit Signals and Systems)
- Entropie `entropy` : J'ai pas vrmt compris / Mesure du désordre ou de l'incertitude dans la distribution des intensités.
- Nombres de pics `num_peaks` : les pics dans la colonne si gros peut-être *fragmented*
- Moyenne du gradiant `mean_gradient` : Moyenne des différences entre pixels consécutifs

#### A faire : 
- Différence Spatiale (un peu comme mes autres méthodes)
- Différence Temporelle (pour les blinking)

In [ ]:
def extract_column_features(image, x):
    column = image[:, x].flatten()  # On force columns à être un vecteur en 1D pour pas avoir d'erreur sur find_peaks

    col_mean = np.mean(column)

    if x == 0: # Si on est sur la toute première colonne à gauche
        neighbor_mean = np.mean(image[:, x+1])
    elif x == image.shape[1] - 1: # Si on est sur la toute dernière colonne à droite
        neighbor_mean = np.mean(image[:, x-1])
    else: # Pour toutes les autres colonnes au milieu
        neighbor_mean = (np.mean(image[:, x-1]) + np.mean(image[:, x+1])) / 2.0
        
    # La Feature Ultime : La différence avec le voisinage
    spatial_diff = abs(col_mean - neighbor_mean)

    # =========================================================================
    # 3. FEATURES EXPERTES INSPIRÉES DE TES MÉTHODES (test_perf_meth)
    # =========================================================================
    
    # A. Inspiré de "local_threshold" (Tendance locale sur fenêtre de 50)
    # Au lieu d'utiliser un seuil strict, on donne au modèle l'écart entre 
    # la colonne et sa zone locale (fenêtre de 50).
    demi_fenetre = 25
    x_min = max(0, x - demi_fenetre)
    x_max = min(image.shape[1], x + demi_fenetre + 1)
    # On calcule la moyenne de cette "fenêtre" autour de la colonne
    tendance_locale_large = np.mean(image[:, x_min:x_max])
    ecart_tendance_50 = np.abs(col_mean - tendance_locale_large)
    
    
    # B. Inspiré de "detecter_et_mesurer_defauts_complet" (Sauts verticaux / Ruptures)
    # Tu utilises la dérivée absolue pour trouver la taille des "marches"
    sauts_verticaux = np.abs(np.diff(column))
    
    bruit_normal_sauts = np.median(sauts_verticaux) # Ce que tu appelais 'bruit_normal'
    ecart_sauts = np.std(sauts_verticaux)
    max_saut = np.max(sauts_verticaux) if len(sauts_verticaux) > 0 else 0
    
    # On crée un "Ratio de Rupture" : Si le saut max est 10x plus grand que le 
    # bruit normal, c'est sûrement une colonne fragmentée ! (Le +1e-5 évite la division par 0)
    ratio_rupture = max_saut / (bruit_normal_sauts + 1e-5)


    return {
        
        'spatial difference' : spatial_diff,
        'mean': col_mean,
        'num_peaks': len(find_peaks(column)[0]),
        'std': np.std(column),
        'min': np.min(column),
        'max': np.max(column),
        'median': np.median(column),
        'q25': np.percentile(column, 25),
        'q75': np.percentile(column, 75),
        'range': np.max(column) - np.min(column),
        'skewness': scipy.stats.skew(column),
        'kurtosis': scipy.stats.kurtosis(column),
        # np.float64 pour éviter l'Integer Overflow
        'energy': np.sum(column.astype(np.float64) ** 2),
        'entropy': scipy.stats.entropy(np.histogram(column, bins=50)[0]),
        'mean_gradient': np.mean(np.gradient(column)), 

        # ====
        'ecart_tendance_50': ecart_tendance_50, # Ton local_threshold
        'mediane_sauts_verticaux': bruit_normal_sauts, # Ta Region Growing
        'std_sauts_verticaux': ecart_sauts,
        'ratio_rupture_verticale': ratio_rupture
    }

In [ ]:
def process_single_image(img_num, image, json_data):
    """Fonction intermédiaire exécutée par chaque cœur du processeur"""
    X_img = []
    y_img = []
    defects = get_defect_coordinates(json_data, img_num)
    
    for x in range(image.shape[1]):
        features = extract_column_features(image, x)
        
        # C'était nul quand je faisais par type de defects
        if x in defects:
            label = 1  # 1 = Defects columns
        else:
            label = 0  # 0 = Sain 
            
        X_img.append(list(features.values()))
        y_img.append(label)
        
    return X_img, y_img

def build_dataset(images, json_data):
    print(f"Lancement de l'extraction sur {len(images)} images en parallèle...")
    
    # n_jobs=-1 (utilisation de tous les coeurs)
    # return_as="generator" permet à tqdm de se mettre à jour en temps réel
    result_generator = Parallel(n_jobs=-1, return_as="generator")(
        delayed(process_single_image)(img_num, img, json_data) 
        for img_num, img in enumerate(images)
    )
    
    X = []
    y = []
    
    # On enveloppe le générateur avec tqdm pour la barre de progression
    for X_img, y_img in tqdm(result_generator, total=len(images), desc="Extraction Multicoeur"):
        X.extend(X_img)
        y.extend(y_img)
        
    return np.array(X), np.array(y)

### Entraînement

In [ ]:
X, y = build_dataset(load_images(sequence='sequence_2', dyn='low dyn with columns 2'), load_json('results/VGA_sequence_2_config_2.json'))

In [ ]:
# --- ÉTAPE DE RÉÉQUILIBRAGE (UNDERSAMPLING) ---
# On récupère les indices des défauts et des colonnes saines
indices_defauts = np.where(y == 1)[0]
indices_sains = np.where(y == 0)[0]

# On choisit aléatoirement autant de colonnes saines qu'il y a de défauts
# (Tu peux multiplier len(indices_defauts) par 3 ou 4 si tu veux un peu plus de saines)
nb_echantillons = len(indices_defauts) * 2 
indices_sains_reduits = np.random.choice(indices_sains, size=nb_echantillons, replace=False)

# On rassemble les indices et on filtre X et y
indices_finaux = np.concatenate([indices_defauts, indices_sains_reduits])
X_balanced = X[indices_finaux]
y_balanced = y[indices_finaux]

print(f"Nouveau Dataset : {len(indices_defauts)} défauts et {len(indices_sains_reduits)} colonnes saines.")

# --- ENTRAÎNEMENT DU MODÈLE ---
X_train, X_test, y_train, y_test = train_test_split(X_balanced, y_balanced, test_size=0.2, random_state=42)

clf = RandomForestClassifier(n_estimators=100, class_weight='balanced', random_state=42)
clf.fit(X_train, y_train)
y_pred = clf.predict(X_test)

print(classification_report(y_test, y_pred))

In [ ]:
# Liste des noms de tes features dans le même ordre que ton dictionnaire
feature_names = ['spatial difference', 'mean', 'num_peaks', 'std', 'min', 'max', 'median', 'q25', 'q75', 'range', 
                 'skewness', 'kurtosis', 'energy', 'entropy', 'mean_gradient', 'ecart_tendance_50',
                 'mediane_sauts_verticaux', 'std_sauts_verticaux', 'ratio_rupture_verticale']

importances = clf.feature_importances_
for name, importance in sorted(zip(feature_names, importances), key=lambda x: x[1], reverse=True):
    print(f"{name}: {importance:.3f}")